# Mobius export → onnx-world-model inference

Complete high-level workflow for `nvidia/Cosmos3-Edge`: text generation, image/video understanding, image/video generation, image-to-video, and action generation.

```powershell
pip install -e ~/workspace/mobius
pip install -e ~/workspace/onnx-world-model
```

The FP32 package is about 23 GB. CPU works but is slow; BF16 with CUDA is recommended.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

from onnx_world_model import WorldModel

EXPORT_DIR = Path("artifacts/cosmos3-edge-f32")
ASSET_DIR = Path("artifacts/inputs")
ASSET_DIR.mkdir(parents=True, exist_ok=True)

## 1. Export every component with one Mobius command

In [ ]:
!mobius build --model nvidia/Cosmos3-Edge {EXPORT_DIR} --features world-model --dtype f32

## 2. Load once

Use `providers=["cuda", "cpu"]` with a CUDA-enabled ONNX Runtime installation.

In [ ]:
model = WorldModel.from_pretrained(EXPORT_DIR, providers=["cpu"])
model.capabilities

Download the small official Edge image and video used by the understanding and I2V examples.

In [ ]:
IMAGE_PATH = ASSET_DIR / "edge_i2v_input.jpg"
VIDEO_PATH = ASSET_DIR / "edge_i2v_output.mp4"
if not IMAGE_PATH.exists(): urlretrieve("https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets/example_i2v_input.jpg", IMAGE_PATH)
if not VIDEO_PATH.exists(): urlretrieve("https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets/diffusers_outputs/edge_i2v_diffusers.mp4", VIDEO_PATH)

## 3. Text generation

In [ ]:
text = model.text.generate("Explain what a world model is in one sentence.", max_tokens=64, do_sample=False)
text.text

## 4. Image understanding

In [ ]:
image_description = model.text.generate("Describe this road scene.", image=IMAGE_PATH, max_tokens=96, do_sample=False)
image_description.text

## 5. Video understanding

In [ ]:
video_description = model.text.generate("Describe what changes in this video.", video=VIDEO_PATH, video_sample_fps=2, max_tokens=128, do_sample=False)
video_description.text

## 6. Text-to-image

In [ ]:
image = model.image.generate("A photorealistic orange cat sitting on a windowsill", height=256, width=256, guidance_scale=5.0, num_inference_steps=50, seed=42)
image.images.shape

## 7. Text-to-video

In [ ]:
text_video = model.video.generate("A robot arm moves a red block to the left", frames=5, height=256, width=256, guidance_scale=5.0, num_inference_steps=50, seed=42)
text_video.video.shape

## 8. Image-to-video

Passing `image=` activates the exported VAE encoder, conditioned latent mask, official Edge scheduler override, and classifier-free guidance.

In [ ]:
image_video = model.video.generate("The car drives along the coastal road", image=IMAGE_PATH, frames=121, height=480, width=832, fps=24, num_inference_steps=50, seed=0)
image_video.video.shape

## 9. Action generation

`domain` selects the exported domain-aware action head. The returned tensor is cropped to that domain's raw action width.

In [ ]:
action = model.action.generate("Move the red block to the left.", domain="droid_lerobot", steps=16, num_inference_steps=30, seed=42)
action.actions.shape

These are all high-level modalities exposed by the Edge package: `text`, `image`, `video`, and `action`. Cosmos3-Edge does not ship the Cosmos3 Sound tokenizer, so audio generation is unavailable for this checkpoint.